In [2]:
# !pip install -r requirements.txt
import importlib
importlib.reload

<function importlib.reload(module)>

In [14]:
!python scripts/convert_dm4.py --input data/Diffraction_SI.dm4 --output data/train_tensor.pt --downsample 8 --mode bin 

Saved 4788 patterns of size 64*64 → data/train_tensor.pt


In [5]:
!python -m scripts.train --data data/train_tensor.npy --epochs 1 --batch 128 --latent 32 --lr 0.00003 --output_dir outputs --device cpu

Seed set to 42
────────────────────────────────────────────────────────────────────────────────
Layer                              Input → Output                    Params
────────────────────────────────────────────────────────────────────────────────
000_Conv2d                         (1, 1, 64, 64) → (1, 16, 32, 32)       160
001_ReLU                           (1, 16, 32, 32) → (1, 16, 32, 32)         0
002_Conv2d                         (1, 16, 32, 32) → (1, 32, 16, 16)     4,640
003_ReLU                           (1, 32, 16, 16) → (1, 32, 16, 16)         0
004_Conv2d                         (1, 32, 16, 16) → (1, 64, 8, 8)    18,496
005_ReLU                           (1, 64, 8, 8) → (1, 64, 8, 8)          0
006_Flatten                        (1, 64, 8, 8) → (1, 4096)              0
007_Linear                         (1, 4096) → (1, 32)              131,104
008_Encoder                        (1, 1, 64, 64) → (1, 32)         154,400
009_Linear                         (1, 32) → (1, 40

In [9]:
!python -m scripts.generate_embeddings --input data/train_tensor_new.npy --checkpoint outputs/ae.ckpt --batch_size 2048 --output outputs/embeddings.pt

────────────────────────────────────────────────────────────────────────────────
Layer                              Input → Output                    Params
────────────────────────────────────────────────────────────────────────────────
000_Conv2d                         (1, 1, 64, 64) → (1, 16, 32, 32)       160
001_ReLU                           (1, 16, 32, 32) → (1, 16, 32, 32)         0
002_Conv2d                         (1, 16, 32, 32) → (1, 32, 16, 16)     4,640
003_ReLU                           (1, 32, 16, 16) → (1, 32, 16, 16)         0
004_Conv2d                         (1, 32, 16, 16) → (1, 64, 8, 8)    18,496
005_ReLU                           (1, 64, 8, 8) → (1, 64, 8, 8)          0
006_Flatten                        (1, 64, 8, 8) → (1, 4096)              0
007_Linear                         (1, 4096) → (1, 32)              131,104
────────────────────────────────────────────────────────────────────────────────
Total params                                                 

In [16]:
import torch, numpy as np
raw = torch.load("data/train_tensor.pt")
print("raw tensor shape:", raw.shape)         # (N, 1, Qy, Qx)
N = raw.shape[0]
print("number of probe positions:", N)

# quick factor search
cands = [(f, N//f) for f in range(1, int(np.sqrt(N))+1) if N % f == 0]
print("factor pairs:", cands[:10], "…")


raw tensor shape: torch.Size([4788, 1, 64, 64])
number of probe positions: 4788
factor pairs: [(1, 4788), (2, 2394), (3, 1596), (4, 1197), (6, 798), (7, 684), (9, 532), (12, 399), (14, 342), (18, 266)] …


In [18]:
# pure square scan, bright‑field background, every latent dim in one mosaic
!python scripts/visualise_scan_latents.py \
       --raw data/train_tensor.pt \
       --latents outputs/embeddings.pt \
       --scan 42 114 \
       --virtual bf \
       --lat_max_cols 6 \
       --outfig outputs/latent_mosaic.png



/Users/louisg/PycharmProjects/m3-learning-m3_learning-dc4a2d5/LTSJ_exp/Custom_4DSTEM_AE/scripts/visualise_scan_latents.py:154: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
Saved outputs/latent_mosaic.png
